# Preprocessing dan Ekstraksi 68 Fitur TSFEL - Tiga Polutan Kecamatan Bangkalan

Notebook ini menjalankan preprocessing dan ekstraksi fitur TSFEL untuk **NO2**,
**CO**, dan **SO2**. Setiap polutan diproses sebagai deret waktu terpisah
dan menghasilkan file fitur wide serta long.

## Tujuan
1. Membaca CSV NO2, CO, dan SO2.
2. Membersihkan data dan mendeteksi outlier dengan IQR.
3. Membentuk indeks tanggal harian lengkap dari 31 Agustus 2025 sampai 31 Agustus 2026.
4. Mengisi missing value menggunakan interpolasi berbasis waktu.
5. Mengekstraksi 68 fitur TSFEL untuk masing-masing polutan.
6. Memeriksa missing value dan infinity pada hasil fitur.
7. Menyimpan hasil fitur wide dan long per polutan.

**Catatan penting:** tanggal tambahan pada indeks harian merupakan hasil imputasi,
bukan observasi Sentinel-5P asli.

In [1]:
import os
import inspect
import numpy as np
import pandas as pd
import tsfel.feature_extraction.features as tsfel_features

In [2]:
POLLUTANTS = ["NO2", "CO", "SO2"]
KECAMATAN_NAME = "Bangkalan"

START_DATE = "2025-08-31"
END_DATE = "2026-08-31"

FS = 1

CSV_DIR = "../data/csv"

INPUT_CSV = {
    pollutant: os.path.join(CSV_DIR, f"{pollutant}-{KECAMATAN_NAME}.csv")
    for pollutant in POLLUTANTS
}
OUTPUT_WIDE = {
    pollutant: os.path.join(CSV_DIR, f"{pollutant}_{KECAMATAN_NAME}_TSFEL_fixed.csv")
    for pollutant in POLLUTANTS
}
OUTPUT_LONG = {
    pollutant: os.path.join(CSV_DIR, f"{pollutant}_{KECAMATAN_NAME}_TSFEL_fixed_long.csv")
    for pollutant in POLLUTANTS
}

print("Polutan:", ", ".join(POLLUTANTS))

Polutan: NO2, CO, SO2


### 1. Memuat dan membersihkan data awal

In [3]:
data_by_pollutant = {}

for pollutant in POLLUTANTS:
    df = pd.read_csv(INPUT_CSV[pollutant])
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df[pollutant] = pd.to_numeric(df[pollutant], errors="coerce")

    df = df.dropna(subset=["date"])
    df = df.sort_values("date")
    df = df.drop_duplicates(subset="date", keep="first")
    df = df[
        (df["date"] >= pd.Timestamp(START_DATE)) &
        (df["date"] <= pd.Timestamp(END_DATE))
    ].reset_index(drop=True)
    data_by_pollutant[pollutant] = df

    print(f"=== KONDISI DATA AWAL {pollutant} ===")
    print(f"Jumlah baris       : {len(df)}")
    print(f"Missing value      : {df[pollutant].isna().sum()}")
    print(f"Tanggal unik       : {df['date'].nunique()}")
    print(f"Tanggal awal       : {df['date'].min().date()}")
    print(f"Tanggal akhir      : {df['date'].max().date()}")

=== KONDISI DATA AWAL NO2 ===
Jumlah baris       : 132
Missing value      : 0
Tanggal unik       : 132
Tanggal awal       : 2025-09-03
Tanggal akhir      : 2026-08-28
=== KONDISI DATA AWAL CO ===
Jumlah baris       : 178
Missing value      : 0
Tanggal unik       : 178
Tanggal awal       : 2025-08-31
Tanggal akhir      : 2026-08-29
=== KONDISI DATA AWAL SO2 ===
Jumlah baris       : 168
Missing value      : 0
Tanggal unik       : 168
Tanggal awal       : 2025-08-31
Tanggal akhir      : 2026-08-28


### 2. Deteksi outlier dengan IQR

In [4]:
for pollutant, df in data_by_pollutant.items():
    Q1 = df[pollutant].quantile(0.25)
    Q3 = df[pollutant].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    is_outlier = (
        (df[pollutant] < lower_bound) |
        (df[pollutant] > upper_bound)
    )
    n_outliers = int(is_outlier.sum())
    df.loc[is_outlier, pollutant] = np.nan

    print(f"=== OUTLIER {pollutant} ===")
    print(f"Q1            : {Q1:.6e}")
    print(f"Q3            : {Q3:.6e}")
    print(f"IQR           : {IQR:.6e}")
    print(f"Batas bawah   : {lower_bound:.6e}")
    print(f"Batas atas    : {upper_bound:.6e}")
    print(f"Jumlah outlier: {n_outliers}")

=== OUTLIER NO2 ===
Q1            : 2.174555e-05
Q3            : 3.719261e-05
IQR           : 1.544706e-05
Batas bawah   : -1.425043e-06
Batas atas    : 6.036320e-05
Jumlah outlier: 4
=== OUTLIER CO ===
Q1            : 2.655426e-02
Q3            : 3.123670e-02
IQR           : 4.682441e-03
Batas bawah   : 1.953060e-02
Batas atas    : 3.826036e-02
Jumlah outlier: 5
=== OUTLIER SO2 ===
Q1            : -7.952048e-05
Q3            : 1.952158e-04
IQR           : 2.747362e-04
Batas bawah   : -4.916249e-04
Batas atas    : 6.073201e-04
Jumlah outlier: 8


### 3. Membentuk deret waktu harian lengkap

Bagian ini adalah perbaikan utama. Kode lama melakukan interpolasi hanya pada tanggal yang sudah ada sehingga jumlah data tetap 132.

In [5]:
full_date_range = pd.date_range(
    start=pd.Timestamp(START_DATE),
    end=pd.Timestamp(END_DATE),
    freq="D"
)

daily_by_pollutant = {}
for pollutant, df in data_by_pollutant.items():
    df_time = df.set_index("date").sort_index()
    df_daily = df_time.reindex(full_date_range)
    df_daily.index.name = "date"
    daily_by_pollutant[pollutant] = df_daily

    missing_before_imputation = int(df_daily[pollutant].isna().sum())
    print(f"{pollutant}: {len(df_daily)} tanggal, {missing_before_imputation} missing sebelum imputasi")

NO2: 366 tanggal, 238 missing sebelum imputasi
CO: 366 tanggal, 193 missing sebelum imputasi
SO2: 366 tanggal, 206 missing sebelum imputasi


### 4. Imputasi missing value

In [6]:
clean_by_pollutant = {}
for pollutant, df_daily in daily_by_pollutant.items():
    df_daily[pollutant] = (
        df_daily[pollutant]
        .interpolate(method="time")
        .ffill()
        .bfill()
    )

    df_clean = df_daily.reset_index()
    df_clean.columns = ["date", pollutant]
    df_clean = df_clean.sort_values("date").reset_index(drop=True)
    clean_by_pollutant[pollutant] = df_clean

    print(f"{pollutant}: {len(df_clean)} baris, {df_clean[pollutant].isna().sum()} missing setelah imputasi")

NO2: 366 baris, 0 missing setelah imputasi
CO: 366 baris, 0 missing setelah imputasi
SO2: 366 baris, 0 missing setelah imputasi


### 5. Verifikasi signal

TSFEL pada notebook lama memberi peringatan karena signal hanya 132 data points, sedangkan fitur fractal membutuhkan minimal 160 data points.

In [7]:
signals_by_pollutant = {}
for pollutant, df_clean in clean_by_pollutant.items():
    signal = df_clean[pollutant].astype(float).values

    if len(signal) < 160:
        raise ValueError(
            f"{pollutant}: panjang signal hanya {len(signal)}. "
            "Fitur fractal TSFEL membutuhkan minimal 160 data points."
        )
    if np.isnan(signal).any():
        raise ValueError(f"{pollutant}: signal masih mengandung NaN.")

    signals_by_pollutant[pollutant] = signal
    print(f"{pollutant}: signal siap, panjang {len(signal)}")

NO2: signal siap, panjang 366
CO: signal siap, panjang 366
SO2: signal siap, panjang 366


### 6. Daftar 68 fitur TSFEL

In [8]:
FEATURE_LIST = """
abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile
ecdf_percentile_count ecdf_slope entropy fundamental_frequency
higuchi_fractal_dimension hist_mode human_range_energy hurst_exponent
interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff
median_abs_deviation median_abs_diff median_diff median_frequency mfcc
mse negative_turning neighbourhood_peaks petrosian_fractal_dimension
pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy
spectral_kurtosis spectral_positive_turning spectral_roll_off
spectral_roll_on spectral_skewness spectral_slope spectral_spread
spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross
""".split()

print("Jumlah fitur:", len(FEATURE_LIST))

Jumlah fitur: 68


### 7. Fungsi ekstraksi

In [9]:
def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]

    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)

        if arr.size == 0:
            return np.nan

        if np.all(np.isnan(arr)):
            return np.nan

        return float(np.nanmean(arr))

    return float(result)


def extract_one(feature_name, signal, fs):
    fn = getattr(tsfel_features, feature_name)
    params = inspect.signature(fn).parameters

    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)

    return to_scalar(result)

### 8. Ekstraksi 68 fitur

In [10]:
extracted_features_by_pollutant = {}

for pollutant, signal in signals_by_pollutant.items():
    row = {}
    print(f"\n=== EKSTRAKSI {pollutant} ===")
    for feature_name in FEATURE_LIST:
        try:
            row[feature_name] = extract_one(feature_name, signal, FS)
            print(f"{feature_name:<35} {row[feature_name]}")
        except Exception as exc:
            print(f"{feature_name:<35} GAGAL: {exc}")
            row[feature_name] = np.nan

    extracted_features_by_pollutant[pollutant] = pd.DataFrame([row])
    print(f"Jumlah kolom hasil {pollutant}: {extracted_features_by_pollutant[pollutant].shape[1]}")


=== EKSTRAKSI NO2 ===
abs_energy                          3.1153029424347104e-07
auc                                 0.010021197456467235
autocorr                            5.0
average_power                       8.535076554615645e-10
calc_centroid                       198.05233188958732
calc_max                            5.502832573256456e-05
calc_mean                           2.7417823733297856e-05
calc_median                         2.6607980241048306e-05
calc_min                            3.7077027172927046e-06
calc_std                            9.971890947142217e-06
calc_var                            9.943860906169691e-11
dfa                                 1.1054594192460196
distance                            365.00000000771587
ecdf                                0.015027322404371584
ecdf_percentile                     2.7339038001267155e-05
ecdf_percentile_count               182.5
ecdf_slope                          34946.039725148716
entropy                           

c:\Users\ASUS\AppData\Local\Programs\Python\Python39\lib\site-packages\tsfel\feature_extraction\features_utils.py:534: RuntimeWarning: invalid value encountered in divide
  rs = np.divide(r, s)


mse                                 1.2209268166361593
negative_turning                    40.0
neighbourhood_peaks                 12.0
petrosian_fractal_dimension         1.014929871984861
pk_pk_distance                      5.1320623015271856e-05
positive_turning                    41.0
power_bandwidth                     0.20491803278688525
rms                                 2.91749150355556e-05
skewness                            0.32699109423078526
slope                               2.07442833874272e-08
spectral_centroid                   0.10911161303646173
spectral_decrease                   -2.322231171469177
spectral_distance                   -1.7807966297524964
spectral_entropy                    0.7468579853152687
spectral_kurtosis                   3.441396614216699
spectral_positive_turning           58.0
spectral_roll_off                   0.4180327868852459
spectral_roll_on                    0.0
spectral_skewness                   1.2599693051902945
spectral_slope  

### 9. Pengelompokan domain

In [11]:
FEATURE_DOMAINS = {
    "statistical": [
        "abs_energy", "average_power", "calc_max", "calc_mean",
        "calc_median", "calc_min", "calc_std", "calc_var",
        "ecdf", "ecdf_percentile", "ecdf_percentile_count",
        "ecdf_slope", "entropy", "hist_mode", "interq_range",
        "kurtosis", "mean_abs_deviation", "median_abs_deviation",
        "pk_pk_distance", "rms", "skewness"
    ],
    "temporal": [
        "auc", "autocorr", "calc_centroid", "distance",
        "lempel_ziv", "mean_abs_diff", "mean_diff",
        "median_abs_diff", "median_diff", "negative_turning",
        "neighbourhood_peaks", "positive_turning", "slope",
        "sum_abs_diff", "zero_cross"
    ],
    "spectral": [
        "fundamental_frequency", "human_range_energy", "lpcc",
        "max_frequency", "max_power_spectrum", "median_frequency",
        "mfcc", "power_bandwidth", "spectral_centroid",
        "spectral_decrease", "spectral_distance", "spectral_entropy",
        "spectral_kurtosis", "spectral_positive_turning",
        "spectral_roll_off", "spectral_roll_on", "spectral_skewness",
        "spectral_slope", "spectral_spread", "spectral_variation",
        "spectrogram_mean_coeff", "wavelet_abs_mean",
        "wavelet_energy", "wavelet_entropy", "wavelet_std",
        "wavelet_var"
    ],
    "fractal": [
        "dfa", "higuchi_fractal_dimension", "hurst_exponent",
        "maximum_fractal_length", "mse",
        "petrosian_fractal_dimension"
    ]
}

feature_domain_map = {
    feature: domain
    for domain, features in FEATURE_DOMAINS.items()
    for feature in features
}

print("Jumlah fitur per domain:")
for domain, features in FEATURE_DOMAINS.items():
    print(f"{domain:12s}: {len(features)}")

print("Total:", sum(len(v) for v in FEATURE_DOMAINS.values()))

Jumlah fitur per domain:
statistical : 21
temporal    : 15
spectral    : 26
fractal     : 6
Total: 68


### 10. Validasi hasil

In [12]:
validation_by_pollutant = {}

for pollutant, extracted_features in extracted_features_by_pollutant.items():
    feature_missing = extracted_features.isna().sum()
    missing_features = feature_missing[feature_missing > 0]
    numeric_result = extracted_features.select_dtypes(include=[np.number])
    infinite_count = int(np.isinf(numeric_result.to_numpy()).sum())
    validation_by_pollutant[pollutant] = {
        "missing_features": missing_features,
        "infinite_count": infinite_count,
    }

    print(f"=== VALIDASI HASIL {pollutant} ===")
    print("Jumlah fitur   :", extracted_features.shape[1])
    print("Jumlah NaN      :", int(extracted_features.isna().sum().sum()))
    print("Jumlah infinity :", infinite_count)
    if len(missing_features) > 0:
        print("Fitur yang masih NaN:")
        print(missing_features)
    else:
        print("Tidak ada missing value pada 68 fitur.")

=== VALIDASI HASIL NO2 ===
Jumlah fitur   : 68
Jumlah NaN      : 0
Jumlah infinity : 0
Tidak ada missing value pada 68 fitur.
=== VALIDASI HASIL CO ===
Jumlah fitur   : 68
Jumlah NaN      : 0
Jumlah infinity : 0
Tidak ada missing value pada 68 fitur.
=== VALIDASI HASIL SO2 ===
Jumlah fitur   : 68
Jumlah NaN      : 0
Jumlah infinity : 0
Tidak ada missing value pada 68 fitur.


In [13]:
for pollutant, extracted_features in extracted_features_by_pollutant.items():
    validation = validation_by_pollutant[pollutant]

    if extracted_features.shape[1] != 68:
        raise ValueError(f"{pollutant}: jumlah fitur bukan 68.")
    if extracted_features.isna().sum().sum() > 0:
        raise ValueError(f"{pollutant}: masih ada NaN pada hasil TSFEL.")
    if validation["infinite_count"] > 0:
        raise ValueError(f"{pollutant}: terdapat nilai infinity pada hasil TSFEL.")

print("SEMUA VALIDASI BERHASIL untuk:", ", ".join(POLLUTANTS))

SEMUA VALIDASI BERHASIL untuk: NO2, CO, SO2


### 11. Membuat format long

In [14]:
features_long_by_pollutant = {}

for pollutant, extracted_features in extracted_features_by_pollutant.items():
    features_long = pd.DataFrame({
        "polutan": pollutant,
        "fitur": list(extracted_features.columns),
        "domain": [
            feature_domain_map.get(feature, "tidak diketahui")
            for feature in extracted_features.columns
        ],
        "nilai": extracted_features.iloc[0].values,
    })
    features_long = (
        features_long
        .sort_values(["domain", "fitur"])
        .reset_index(drop=True)
    )
    features_long_by_pollutant[pollutant] = features_long

features_long_by_pollutant["NO2"].head()

,polutan,fitur,domain,nilai
0,NO2,dfa,fractal,1.105459
1,NO2,higuchi_fractal_dimension,fractal,1.760954
2,NO2,hurst_exponent,fractal,0.893096
3,NO2,maximum_fractal_length,fractal,-2.703718
4,NO2,mse,fractal,1.220927


### 12. Menyimpan hasil ke CSV

In [15]:
for pollutant in POLLUTANTS:
    extracted_features_by_pollutant[pollutant].to_csv(
        OUTPUT_WIDE[pollutant],
        index=False,
    )
    features_long_by_pollutant[pollutant].to_csv(
        OUTPUT_LONG[pollutant],
        index=False,
    )

    print(f"{pollutant} wide : {OUTPUT_WIDE[pollutant]}")
    print(f"{pollutant} long : {OUTPUT_LONG[pollutant]}")

NO2 wide : NO2_Bangkalan_TSFEL_fixed.csv
NO2 long : NO2_Bangkalan_TSFEL_fixed_long.csv
CO wide : CO_Bangkalan_TSFEL_fixed.csv
CO long : CO_Bangkalan_TSFEL_fixed_long.csv
SO2 wide : SO2_Bangkalan_TSFEL_fixed.csv
SO2 long : SO2_Bangkalan_TSFEL_fixed_long.csv


---

# Ringkasan Proses

Notebook ini memproses tiga deret waktu polutan Kecamatan Bangkalan:
**NO2**, **CO**, dan **SO2**.

Untuk setiap polutan, prosesnya adalah:
1. Membaca CSV hasil ekstraksi Sentinel-5P.
2. Mendeteksi outlier menggunakan IQR dan menandainya sebagai missing value.
3. Membentuk tanggal harian lengkap dari 31 Agustus 2025 sampai 31 Agustus 2026.
4. Mengisi missing value dengan interpolasi berbasis waktu.
5. Mengekstraksi dan memvalidasi 68 fitur TSFEL.
6. Menyimpan hasil dalam format wide dan long.

Output yang dihasilkan untuk setiap polutan:
- `<POLUTAN>_Bangkalan_TSFEL_fixed.csv`: satu baris dengan 68 fitur.
- `<POLUTAN>_Bangkalan_TSFEL_fixed_long.csv`: 68 baris dengan kolom `polutan`, `fitur`, `domain`, dan `nilai`.

**Catatan metodologis:** tanggal tambahan pada deret harian adalah hasil imputasi,
bukan observasi Sentinel-5P asli. Hal ini perlu disebutkan dalam laporan.

---

# 4. Penggabungan 3 Polutan Menjadi 204 Kolom (Linear dan Polynomial)

Langkah berikut mengimplementasikan tugas penggabungan 3 polutan:
1. Menggabungkan deret waktu tiga polutan (**NO2**, **SO2**, **CO**) Kecamatan Bangkalan ke dalam satu tabel harian lengkap (31 Agustus 2025 - 31 Agustus 2026).
2. Menangani pencilan (*outliers*) menggunakan batas IQR ($Q1 - 1.5 \times IQR$ dan $Q3 + 1.5 \times IQR$) dengan menjadikannya `NaN`.
3. Melakukan imputasi menggunakan dua metode interpolasi:
   - **Linear / Time-based Interpolation** (`interpolate(method='time')`)
   - **Polynomial Interpolation** (`interpolate(method='polynomial', order=2)`)
4. Mengekstraksi **68 fitur TSFEL** per polutan dengan penamaan berawalan polutan (`<POLUTAN>_<FITUR>`), menghasilkan **1 baris $\times$ 204 kolom**.
5. Menyimpan ke dua berkas CSV di folder `data/csv/`:
   - `All-Pollutants-Bangkalan-TSFEL-linear.csv`
   - `All-Pollutants-Bangkalan-TSFEL-polynomial.csv`


In [ ]:
import inspect
import numpy as np
import pandas as pd
import tsfel.feature_extraction.features as tsfel_features

# ---------- 1. Muat dan gabungkan data 3 polutan Kecamatan Bangkalan ----------
CSV_DIR = "../data/csv"

df_no2 = pd.read_csv(f"{CSV_DIR}/NO2-Bangkalan.csv")
df_co  = pd.read_csv(f"{CSV_DIR}/CO-Bangkalan.csv")
df_so2 = pd.read_csv(f"{CSV_DIR}/SO2-Bangkalan.csv")

for d in [df_no2, df_co, df_so2]:
    d["date"] = pd.to_datetime(d["date"])

# Gabungkan berdasarkan tanggal
df_merged = df_no2.merge(df_co, on="date", how="outer").merge(df_so2, on="date", how="outer")
df_merged = df_merged.sort_values("date").reset_index(drop=True)

# Indeks tanggal harian lengkap
full_dates = pd.date_range(start="2025-08-31", end="2026-08-31", freq="D")
df_main = pd.DataFrame({"date": full_dates}).merge(df_merged, on="date", how="left")

pollutants = ["NO2", "SO2", "CO"]
fs = 1

# ---------- 2. Daftar 68 fitur TSFEL ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print(f"Jumlah fitur per polutan: {len(FEATURE_LIST)}")
print(f"Total target fitur keseluruhan: {len(FEATURE_LIST) * len(pollutants)}")

def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)

def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)

# ---------- 3. Ekstraksi untuk Linear dan Polynomial Interpolation ----------
for method_name, interp_method in [("linear", "time"), ("polynomial", "polynomial")]:
    print(f"\n=== Ekstraksi 204 Fitur (Metode: {method_name.upper()}) ===")
    combined_row = {}
    for pollutant in pollutants:
        df_poly = df_main[["date", pollutant]].copy()
        df_poly[pollutant] = pd.to_numeric(df_poly[pollutant], errors="coerce")
        n_missing_before = df_poly[pollutant].isna().sum()
        
        # Handling outlier dengan IQR
        Q1 = df_poly[pollutant].quantile(0.25)
        Q3 = df_poly[pollutant].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df_poly.loc[(df_poly[pollutant] < lower_bound) | (df_poly[pollutant] > upper_bound), pollutant] = np.nan
        
        # Imputasi waktu / polynomial
        if interp_method == "polynomial":
            df_clean = df_poly.set_index("date").interpolate(method="polynomial", order=2).ffill().bfill()
        else:
            df_clean = df_poly.set_index("date").interpolate(method="time").ffill().bfill()
            
        signal_1d = df_clean[pollutant].astype(float).values
        
        # Ekstraksi fitur dan prefix nama polutan (misal: NO2_abs_energy)
        for fn_name in FEATURE_LIST:
            feature_key = f"{pollutant}_{fn_name}"
            combined_row[feature_key] = extract_one(fn_name, signal_1d, fs)
            
    extracted_final = pd.DataFrame([combined_row])
    output_filename = f"{CSV_DIR}/All-Pollutants-Bangkalan-TSFEL-{method_name}.csv"
    extracted_final.to_csv(output_filename, index=False)
    print(f"Berhasil! Total kolom: {extracted_final.shape[1]}, Disimpan -> {output_filename}")
